# Preprocessing — Van Zieleghem, Dai Pra

## Table des matières
1. [Imports](#imports)
2. [Chargement des données](#chargement)
3. [Feature Engineering](#feature-engineering)
4. [Regroupement des titres rares](#titres-rares)
5. [Séparation cible / features](#separation)
6. [Train / Test Split](#split)
7. [Gestion des valeurs manquantes](#missing)
8. [Encodage des variables catégorielles](#encodage)
9. [Standardisation des variables numériques](#standardisation)
10. [Concaténation et sauvegarde](#sauvegarde)

## 1. Imports <a id='imports'></a>

On importe les librairies nécessaires au preprocessing : pandas et numpy pour la manipulation de données, sklearn pour le découpage train/test, l'encodage et la standardisation.

In [1]:
# librairies de manipulation de données
import pandas as pd
import numpy as np

# découpage train/test
from sklearn.model_selection import train_test_split

# encodage des variables catégorielles
from sklearn.preprocessing import OneHotEncoder

# standardisation des variables numériques
from sklearn.preprocessing import StandardScaler

# pour ignorer les warnings non-critiques
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

Les librairies sont importées. On utilise `OneHotEncoder` pour les variables catégorielles et `StandardScaler` pour normaliser les variables numériques.

## 2. Chargement des données <a id='chargement'></a>

On charge le dataset Titanic depuis le fichier CSV présent dans le même répertoire.

In [2]:
# chargement du dataset
df_data = pd.read_csv('Titanic Dataset.csv')

print("Dimensions du dataset :", df_data.shape)
print("\nAperçu des premières lignes :")
df_data.head()

Dimensions du dataset : (1309, 11)

Aperçu des premières lignes :


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.00,0,0,24160,211.3375,B5,S
1,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.5500,C22 C26,S
2,1,0,"Allison, Miss. Helen Loraine",female,2.00,1,2,113781,151.5500,C22 C26,S
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.00,1,2,113781,151.5500,C22 C26,S
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.00,1,2,113781,151.5500,C22 C26,S


Le dataset contient **1309 passagers** et **11 variables**. On retrouve les informations classiques : classe, nom, sexe, âge, famille, tarif, cabine, port d'embarquement, et la variable cible `survived`.

## 3. Feature Engineering <a id='feature-engineering'></a>

On crée de nouvelles variables à partir des données existantes, identifiées comme pertinentes lors de l'analyse exploratoire (Phase 1) :
- **Title** : titre social extrait du nom (Mr, Mrs, Miss, Master, etc.)
- **FamilySize** : taille de la famille à bord (sibsp + parch + 1)
- **IsAlone** : indicateur binaire — le passager voyage-t-il seul ?
- **HasCabin** : indicateur binaire — le numéro de cabine est-il renseigné ?

In [3]:
# extraction du titre social depuis le nom (ex: "Braund, Mr. Owen" -> "Mr")
df_data['Title'] = df_data['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# taille de la famille
df_data['FamilySize'] = df_data['sibsp'] + df_data['parch'] + 1

# voyageur seul
df_data['IsAlone'] = (df_data['FamilySize'] == 1).astype(int)

# cabine renseignée ou non
df_data['HasCabin'] = df_data['cabin'].notnull().astype(int)

print("Nouvelles colonnes créées :", ['Title', 'FamilySize', 'IsAlone', 'HasCabin'])
print("\nDistribution des titres :")
print(df_data['Title'].value_counts())

Nouvelles colonnes créées : ['Title', 'FamilySize', 'IsAlone', 'HasCabin']

Distribution des titres :
Title
Mr          757
Miss        260
Mrs         197
Master       61
Dr            8
Rev           8
Col           4
Major         2
Mlle          2
Ms            2
Lady          1
Capt          1
Mme           1
Sir           1
Jonkheer      1
Dona          1
Don           1
Countess      1
Name: count, dtype: int64


On observe 18 titres différents. Certains sont très rares (Jonkheer, Countess, Dona, etc.) et devront être regroupés pour éviter des colonnes quasi-vides après encodage. Les variables `FamilySize`, `IsAlone` et `HasCabin` capturent des informations indirectes sur le statut social et les chances de survie.

## 4. Regroupement des titres rares <a id='titres-rares'></a>

On regroupe les titres rares en une catégorie `Rare`, et on harmonise les équivalents français/anglais (Mlle → Miss, Mme → Mrs, Ms → Miss).

In [4]:
# regroupement des titres rares et harmonisation
title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Capt': 'Rare', 'Sir': 'Rare',
    'Jonkheer': 'Rare', 'Dona': 'Rare', 'Don': 'Rare', 'Countess': 'Rare'
}

df_data['Title'] = df_data['Title'].map(title_map)

# si des titres non mappés restent (NaN), on les met en Rare
df_data['Title'] = df_data['Title'].fillna('Rare')

print("Distribution des titres après regroupement :")
print(df_data['Title'].value_counts())

Distribution des titres après regroupement :
Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64


Après regroupement, on obtient 5 catégories de titres : Mr, Miss, Mrs, Master et Rare. C'est un nombre gérable pour l'encodage OneHot — cela créera 5 colonnes au lieu de 18.

## 5. Séparation cible / features <a id='separation'></a>

On sépare la variable cible (`survived`) des features. On supprime les colonnes non-exploitables en l'état : `name`, `ticket` et `cabin` (déjà capturé via `HasCabin` et `Title`).

In [5]:
# séparation de la variable cible
y = df_data['survived'].to_numpy()

# suppression des colonnes inutiles pour la modélisation
df_data = df_data.drop(['survived', 'name', 'ticket', 'cabin'], axis=1)

print("Shape des features :", df_data.shape)
print("Colonnes restantes :", list(df_data.columns))
print("\nShape de la cible :", y.shape)
print("Distribution de la cible :", np.bincount(y))

Shape des features : (1309, 11)
Colonnes restantes : ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin']

Shape de la cible : (1309,)
Distribution de la cible : [809 500]


On conserve les colonnes : `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, `embarked`, `Title`, `FamilySize`, `IsAlone`, `HasCabin`. La variable cible `survived` est un vecteur 1D de 0 et 1.

## 6. Train / Test Split <a id='split'></a>

On découpe le dataset en 80% train et 20% test avec `random_state=0` pour assurer la reproductibilité.

In [6]:
# découpage train/test (80/20)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    df_data, y, test_size=0.2, random_state=0
)

print("X_train :", X_train_df.shape)
print("X_test  :", X_test_df.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (1047, 11)
X_test  : (262, 11)
y_train : (1047,)
y_test  : (262,)


Le split donne environ 1047 exemples d'entraînement et 262 de test. On effectue le split **avant** l'imputation des valeurs manquantes pour éviter le data leakage (les statistiques d'imputation ne doivent être calculées que sur le train).

## 7. Gestion des valeurs manquantes <a id='missing'></a>

Le dataset contient des valeurs manquantes dans `age` (~20%), `fare` (1 valeur) et `embarked` (2 valeurs). On impute :
- **age** et **fare** : par la **médiane** du train (robuste aux outliers)
- **embarked** : par le **mode** du train (valeur la plus fréquente)

Important : on calcule les statistiques sur le train uniquement, puis on les applique au train ET au test.

In [7]:
# affichage des valeurs manquantes avant imputation
print("Valeurs manquantes AVANT imputation :")
print("Train :")
print(X_train_df.isnull().sum()[X_train_df.isnull().sum() > 0])
print("\nTest :")
print(X_test_df.isnull().sum()[X_test_df.isnull().sum() > 0])

# calcul des statistiques d'imputation sur le train uniquement
age_median = X_train_df['age'].median()
fare_median = X_train_df['fare'].median()
embarked_mode = X_train_df['embarked'].mode()[0]

print(f"\nImputation — age: {age_median}, fare: {fare_median}, embarked: '{embarked_mode}'")

# application de l'imputation sur train ET test
X_train_df = X_train_df.copy()
X_test_df = X_test_df.copy()

X_train_df['age'] = X_train_df['age'].fillna(age_median)
X_train_df['fare'] = X_train_df['fare'].fillna(fare_median)
X_train_df['embarked'] = X_train_df['embarked'].fillna(embarked_mode)

X_test_df['age'] = X_test_df['age'].fillna(age_median)
X_test_df['fare'] = X_test_df['fare'].fillna(fare_median)
X_test_df['embarked'] = X_test_df['embarked'].fillna(embarked_mode)

print("\nValeurs manquantes APRÈS imputation :")
print("Train :", X_train_df.isnull().sum().sum())
print("Test  :", X_test_df.isnull().sum().sum())

Valeurs manquantes AVANT imputation :
Train :
age         206
fare          1
embarked      2
dtype: int64

Test :
age    57
dtype: int64

Imputation — age: 28.0, fare: 14.4542, embarked: 'S'

Valeurs manquantes APRÈS imputation :
Train : 0
Test  : 0


Toutes les valeurs manquantes ont été imputées. La médiane est préférée à la moyenne pour `age` et `fare` car ces variables contiennent des outliers (tarifs très élevés). Le mode est utilisé pour `embarked` car c'est une variable catégorielle. Les statistiques sont calculées uniquement sur le train pour éviter le data leakage.

## 8. Encodage des variables catégorielles <a id='encodage'></a>

On applique un encodage **OneHot** sur les variables catégorielles : `sex`, `embarked` et `Title`. Cela transforme chaque catégorie en une colonne binaire (0/1).

In [8]:
# identification des colonnes catégorielles et numériques
cat_cols = ['sex', 'embarked', 'Title']
num_cols = [col for col in X_train_df.columns if col not in cat_cols]

print("Colonnes catégorielles :", cat_cols)
print("Colonnes numériques :", num_cols)

# extraction des sous-ensembles catégoriels
X_train_cat = X_train_df[cat_cols].to_numpy()
X_test_cat = X_test_df[cat_cols].to_numpy()

# encodage OneHot (fit sur train, transform sur train et test)
one_hot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_encoded = one_hot.fit_transform(X_train_cat)
X_test_encoded = one_hot.transform(X_test_cat)

cat_feature_names = list(one_hot.get_feature_names_out(cat_cols))
print("\nFeatures après encodage :", cat_feature_names)
print("Shape train encodé :", X_train_encoded.shape)
print("Shape test encodé  :", X_test_encoded.shape)

Colonnes catégorielles : ['sex', 'embarked', 'Title']
Colonnes numériques : ['pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'HasCabin']

Features après encodage : ['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare']
Shape train encodé : (1047, 10)
Shape test encodé  : (262, 10)


L'encodage OneHot a créé des colonnes binaires pour chaque catégorie : `sex_female`, `sex_male`, `embarked_C`, `embarked_Q`, `embarked_S`, et les 5 catégories de titres. Le paramètre `handle_unknown='ignore'` assure la robustesse si le test contient des catégories non vues dans le train.

## 9. Standardisation des variables numériques <a id='standardisation'></a>

On standardise les variables numériques (moyenne=0, écart-type=1) avec `StandardScaler`. Cela est essentiel pour les algorithmes sensibles à l'échelle comme KNN. Le scaler est fit uniquement sur le train.

In [9]:
# extraction des sous-ensembles numériques
X_train_num = X_train_df[num_cols].to_numpy()
X_test_num = X_test_df[num_cols].to_numpy()

# standardisation (fit sur train uniquement)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

print("Colonnes numériques standardisées :", num_cols)
print("Shape train numérique :", X_train_scaled.shape)
print("Shape test numérique  :", X_test_scaled.shape)

Colonnes numériques standardisées : ['pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'HasCabin']
Shape train numérique : (1047, 8)
Shape test numérique  : (262, 8)


Les variables numériques (`pclass`, `age`, `sibsp`, `parch`, `fare`, `FamilySize`, `IsAlone`, `HasCabin`) sont maintenant centrées-réduites. Note : `IsAlone` et `HasCabin` sont binaires (0/1) — leur standardisation ne pose pas de problème mais modifie peu leur distribution.

## 10. Concaténation et sauvegarde <a id='sauvegarde'></a>

On concatène les variables catégorielles encodées et les variables numériques standardisées, puis on sauvegarde les 4 fichiers CSV requis.

In [10]:
# fonction utilitaire
def list_to_string(liste):
    """Convertit une liste en chaîne séparée par des virgules."""
    return ",".join(str(e) for e in liste)

# concaténation : encodé (catégoriel) + standardisé (numérique)
X_train_final = np.concatenate((X_train_encoded, X_train_scaled), axis=1)
X_test_final = np.concatenate((X_test_encoded, X_test_scaled), axis=1)

# noms des colonnes finales
feature_names = cat_feature_names + num_cols
header_str = list_to_string(feature_names)

print("Shape finale X_train :", X_train_final.shape)
print("Shape finale X_test  :", X_test_final.shape)
print("Colonnes finales :", feature_names)

# sauvegarde des fichiers CSV standardisés (utilisés par P, KB et la modélisation)
np.savetxt("X_train.csv", X_train_final, delimiter=",", fmt="%.4f", header=header_str, comments="")
np.savetxt("X_test.csv", X_test_final, delimiter=",", fmt="%.4f", header=header_str, comments="")

np.savetxt("y_train.csv", y_train, delimiter=",", fmt="%d", header="survived", comments="")
np.savetxt("y_test.csv", y_test, delimiter=",", fmt="%d", header="survived", comments="")

# sauvegarde aussi des versions NON standardisées (utilisées uniquement par VarianceThreshold)
# encodé (OneHot, déjà 0/1) + numérique brut (non scalé)
X_train_brut = np.concatenate((X_train_encoded, X_train_num), axis=1)
X_test_brut  = np.concatenate((X_test_encoded,  X_test_num),  axis=1)

np.savetxt("X_train_brut.csv", X_train_brut, delimiter=",", fmt="%.4f", header=header_str, comments="")
np.savetxt("X_test_brut.csv",  X_test_brut,  delimiter=",", fmt="%.4f", header=header_str, comments="")

print("6 fichiers CSV sauvegardés avec succès :")
print(f"  - X_train.csv      ({X_train_final.shape[0]} x {X_train_final.shape[1]}) [standardisé]")
print(f"  - X_test.csv       ({X_test_final.shape[0]} x {X_test_final.shape[1]}) [standardisé]")
print(f"  - X_train_brut.csv ({X_train_brut.shape[0]} x {X_train_brut.shape[1]}) [non standardisé, pour VT]")
print(f"  - X_test_brut.csv  ({X_test_brut.shape[0]} x {X_test_brut.shape[1]}) [non standardisé, pour VT]")
print(f"  - y_train.csv ({y_train.shape[0]} valeurs)")
print(f"  - y_test.csv ({y_test.shape[0]} valeurs)")

Shape finale X_train : (1047, 18)
Shape finale X_test  : (262, 18)
Colonnes finales : ['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'HasCabin']
6 fichiers CSV sauvegardés avec succès :
  - X_train.csv      (1047 x 18) [standardisé]
  - X_test.csv       (262 x 18) [standardisé]
  - X_train_brut.csv (1047 x 18) [non standardisé, pour VT]
  - X_test_brut.csv  (262 x 18) [non standardisé, pour VT]
  - y_train.csv (1047 valeurs)
  - y_test.csv (262 valeurs)


Les 6 fichiers CSV sont sauvegardés avec leurs en-têtes. Le preprocessing est terminé :
- Les valeurs manquantes ont été imputées (médiane / mode)
- Les titres rares ont été regroupés
- Les variables catégorielles sont encodées en OneHot
- Les variables numériques sont standardisées
- Aucun data leakage : toutes les transformations sont fit sur le train uniquement

**Fichiers standardisés** (`X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`) : utilisés par la sélection Personnelle (P), SelectKBest (KB) et tous les notebooks de modélisation.

**Fichiers bruts** (`X_train_brut.csv`, `X_test_brut.csv`) : utilisés **uniquement** par `VarianceThreshold` dans le notebook de sélection, afin que le filtre de variance soit calculé sur des valeurs à leur échelle naturelle (et non après standardisation où toutes les variances seraient ≈ 1).